In [ ]:
import pandas as pd
Emotions = pd.read_csv('emotion.csv')
Emotions.head()

In [ ]:
Emotions=Emotions.fillna(0)
print(Emotions.isna().any())
Emotions.shape

In [ ]:
import numpy as np
np.sum(Emotions.isna())

In [ ]:
X = Emotions.iloc[: ,:-1].values
Y = Emotions['Emotions'].values

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
encoder = OneHotEncoder()
Y = encoder.fit_transform(np.array(Y).reshape(-1,1)).toarray()

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, random_state=42,test_size=0.2, shuffle=True)
x_train.shape, y_train.shape, x_test.shape, y_test.shape

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# 1. Convert one-hot encoded labels to class indices
y_train_labels = np.argmax(y_train, axis=1)  # shape: (30678,)
y_test_labels = np.argmax(y_test, axis=1)    # shape: (7670,)

# 2. Convert to PyTorch tensors
x_train_tensor = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)  # (N, C=1, L)

y_train_tensor = torch.tensor(y_train_labels, dtype=torch.long)

x_test_tensor = torch.tensor(x_test, dtype=torch.float32).unsqueeze(1)

y_test_tensor = torch.tensor(y_test_labels, dtype=torch.long)

# 3. Create datasets
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

# 4. Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import snntorch as snn
from snntorch import surrogate
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

class CNN1D(nn.Module):
    def __init__(self, input_size=2376, num_classes=5):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, stride=1, padding=2)
        self.pool1 = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2)
        self.pool2 = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        
        # Calculate flattened size
        self.flat_size = self._get_flat_size(input_size)
        
        self.fc1 = nn.Linear(self.flat_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def _get_flat_size(self, input_size):
        dummy = torch.zeros(1, 1, input_size)
        dummy = self.pool1(F.relu(self.conv1(dummy)))
        dummy = self.pool2(F.relu(self.conv2(dummy)))
        return dummy.view(1, -1).size(1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class CSNN1D(nn.Module):
    def __init__(self, cnn, beta=0.9, spike_grad=surrogate.fast_sigmoid(), num_steps=25):
        super(CSNN1D, self).__init__()
        self.num_steps = num_steps
        
        # Convert CNN layers to spiking
        self.conv1 = cnn.conv1
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.pool1 = cnn.pool1
        
        self.conv2 = cnn.conv2
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.pool2 = cnn.pool2
        
        self.fc1 = cnn.fc1
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        
        self.fc2 = cnn.fc2

    def forward(self, x):
        # Initialize membrane potentials
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        
        # Record output spikes
        spk_out_rec = []
        
        # Time-step simulation
        for step in range(self.num_steps):
            cur = x[step]  # [batch, 1, 2376]
            
            cur = self.conv1(cur)  # [batch, 16, 2376]
            spk1, mem1 = self.lif1(cur, mem1)
            cur = self.pool1(spk1)  # [batch, 16, 1188]
            
            cur = self.conv2(cur)  # [batch, 32, 1188]
            spk2, mem2 = self.lif2(cur, mem2)
            cur = self.pool2(spk2)  # [batch, 32, 594]
            
            cur = cur.view(cur.size(0), -1)  # Flatten
            cur = self.fc1(cur)  # [batch, 128]
            spk3, mem3 = self.lif3(cur, mem3)
            
            cur = self.fc2(spk3)  # [batch, 5]
            spk_out_rec.append(cur)
        
        return torch.stack(spk_out_rec, dim=0)  # [num_steps, batch, 5]

def rate_encoding_1d(vectors, num_steps=25, max_rate=1.0):
    """
    Convert 1D vectors to spike trains using rate-based encoding
    vectors: [batch, 1, length]
    Returns: [num_steps, batch, 1, length]
    """
    batch_size, channels, length = vectors.shape
    
    # Normalize to [0, 1] per sample
    min_vals = vectors.min(dim=2, keepdim=True)[0]
    max_vals = vectors.max(dim=2, keepdim=True)[0]
    vectors = (vectors - min_vals) / (max_vals - min_vals + 1e-8)
    
    # Generate spike trains
    spike_trains = torch.zeros(num_steps, batch_size, channels, length)
    for t in range(num_steps):
        probs = vectors * max_rate
        spike_trains[t] = (torch.rand_like(vectors) < probs).float()
    
    return spike_trains

def train_cnn(model, train_loader, device, epochs=10, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    model.to(device)
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(output, 1)
            correct += (predicted == target).sum().item()
            total += target.size(0)
        
        accuracy = 100 * correct / total
        print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%')

def evaluate_csnn(model, test_loader, device, num_steps=25):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            spike_data = rate_encoding_1d(data, num_steps).to(device)
            outputs = model(spike_data)  # [num_steps, batch, 5]
            outputs = outputs.mean(dim=0)  # Average over time
            _, predicted = torch.max(outputs, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    
    accuracy = 100 * correct / total
    print(f'CSNN Accuracy: {accuracy:.2f}%')
    return accuracy

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Create and train CNN
    cnn = CNN1D().to(device)
    print("Training CNN...")
    train_cnn(cnn, train_loader, device, epochs=10)
    
    # Create CSNN from trained CNN
    csnn = CSNN1D(cnn).to(device)
    print("Evaluating CSNN...")
    accuracy = evaluate_csnn(csnn, test_loader, device, num_steps=100)
    print(f"CSNN Accuracy: {accuracy:.2f}%")